# Induction Motor Fault Detection - RMS Model V2

This notebook is self-contained for Colab. It improves the RMS-based model by using a fixed 1000-sample cycle, time-based train/validation/test splits inside every CSV file, phase-swapped augmentation, richer RMS imbalance features, confidence output, and a rolling decision helper for Raspberry Pi deployment.

In [1]:
!pip install xgboost==2.0.3 pandas scikit-learn joblib tqdm --quiet
print("Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.1/297.1 MB 4.5 MB/s eta 0:00:00
Dependencies installed


In [2]:
import json
import os
import time
import warnings
from collections import Counter, deque

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Your raw CSV sampling time is 2.0e-5 s and supply frequency is 50 Hz.
# So one electrical cycle = 0.02 s = 1000 samples.
SAMPLES_PER_CYCLE = 1000
SAMPLE_INTERVAL_S = 2.0e-5
LINE_FREQUENCY_HZ = 50.0

TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

RMS_DATASET_PATH = "processed_data_v2/rms_cycle_dataset_v2.csv"
MODEL_DIR = "trained_models_v2"

CSV_FILES = [
    "Fullload_1_mu_rf3.csv",
    "Fullload_3_mu_Rf3.csv",
    "Fullload_5_mu_Rf3.csv",
    "Fullload_healthy.csv",
    "Halfload_1_mu_Rf3.csv",
    "Halfload_3_mu_Rf3.csv",
    "Halfload_5__rf3.csv",
    "Halfload_healthy.csv",
    "Noload_1_mu_Rf3.csv",
    "Noload_3_mu_Rf3.csv",
    "Noload_5_mu_Rf3.csv",
    "Noload_healthy.csv",
]

BINARY_LABELS = {0: "Healthy", 1: "Faulty"}
SEVERITY_LABELS = {0: "1u", 1: "3u", 2: "5u"}
PHASE_LABELS = {0: "Phase 1", 1: "Phase 2", 2: "Phase 3"}
LOAD_LABELS = {0: "No Load", 1: "Half Load", 2: "Full Load"}

FEATURE_NAMES = [
    "I1_rms", "I2_rms", "I3_rms",
    "I_sum", "I_mean", "I_abs_max", "I_abs_min", "I_range", "I_std",
    "imbalance_percent",
    "I1_to_mean", "I2_to_mean", "I3_to_mean",
    "I1_minus_I2", "I2_minus_I3", "I3_minus_I1",
]

In [3]:
def check_gpu():
    try:
        test_model = xgb.XGBClassifier(tree_method="hist", device="cuda", n_estimators=1, verbosity=0)
        test_model.fit(np.random.rand(10, 3), np.random.randint(0, 2, 10))
        print("GPU detected and working")
        return True
    except Exception as exc:
        print(f"GPU not available, using CPU: {exc}")
        return False


def extract_labels(filename):
    fn = filename.lower()
    is_faulty = 0 if "healthy" in fn else 1

    # Severity is only meaningful for faulty samples.
    # For model training, faulty severities are mapped to 0,1,2.
    severity_class = -1
    if is_faulty:
        if "_1_" in fn or "1_mu" in fn:
            severity_class = 0
        elif "_3_" in fn or "3_mu" in fn:
            severity_class = 1
        elif "_5_" in fn or "_5__" in fn:
            severity_class = 2
        else:
            raise ValueError(f"Could not infer severity from {filename}")

    if "fullload" in fn:
        load_class = 2
    elif "halfload" in fn:
        load_class = 1
    elif "noload" in fn:
        load_class = 0
    else:
        raise ValueError(f"Could not infer load from {filename}")

    return is_faulty, severity_class, load_class


def find_current_cols(columns):
    cols = list(columns)
    i1 = next((c for c in cols if "i1" in c.lower()), None)
    i2 = next((c for c in cols if "i2" in c.lower()), None)
    i3 = next((c for c in cols if "i3" in c.lower()), None)
    if i1 and i2 and i3:
        return [i1, i2, i3]
    raise ValueError(f"Cannot find I1, I2, I3 in columns: {cols}")


def assign_time_split(cycle_index, total_cycles):
    train_end = int(total_cycles * TRAIN_FRACTION)
    val_end = int(total_cycles * (TRAIN_FRACTION + VAL_FRACTION))
    if cycle_index < train_end:
        return "train"
    if cycle_index < val_end:
        return "val"
    return "test"


def compute_rms_per_cycle(df, current_cols):
    numeric_df = df[current_cols].apply(pd.to_numeric, errors="coerce")
    values = numeric_df.to_numpy(dtype=np.float32)
    valid_rows = np.isfinite(values).all(axis=1)
    values = values[valid_rows]

    total_cycles = len(values) // SAMPLES_PER_CYCLE
    if total_cycles == 0:
        raise ValueError("Not enough valid samples to form one complete cycle")

    used_samples = total_cycles * SAMPLES_PER_CYCLE
    trimmed = values[:used_samples]
    cycles = trimmed.reshape(total_cycles, SAMPLES_PER_CYCLE, 3)
    rms_values = np.sqrt(np.mean(np.square(cycles), axis=1))

    cycle_df = pd.DataFrame(rms_values, columns=["I1_rms", "I2_rms", "I3_rms"])
    cycle_df["cycle_index"] = np.arange(total_cycles, dtype=np.int32)
    cycle_df["raw_rows_used"] = used_samples
    cycle_df["raw_rows_dropped"] = len(values) - used_samples
    return cycle_df


def build_rms_dataset(csv_files, output_csv=RMS_DATASET_PATH):
    print("\n" + "=" * 70)
    print("BUILDING FIXED-CYCLE RMS DATASET")
    print("=" * 70)

    rows = []
    summaries = []

    for fpath in tqdm(csv_files, desc="Converting raw CSVs to RMS cycles"):
        if not os.path.exists(fpath):
            print(f"Missing file: {fpath}")
            continue

        filename = os.path.basename(fpath)
        is_faulty, severity_class, load_class = extract_labels(filename)
        df = pd.read_csv(fpath, on_bad_lines="skip")
        current_cols = find_current_cols(df.columns)
        cycle_df = compute_rms_per_cycle(df, current_cols)
        total_cycles = len(cycle_df)

        cycle_df["source_file"] = filename
        cycle_df["binary"] = is_faulty
        cycle_df["severity"] = severity_class
        cycle_df["load"] = load_class
        cycle_df["phase"] = -1
        cycle_df["split"] = [assign_time_split(i, total_cycles) for i in cycle_df["cycle_index"]]
        cycle_df["samples_per_cycle"] = SAMPLES_PER_CYCLE
        cycle_df["sample_interval_s"] = SAMPLE_INTERVAL_S
        cycle_df["line_frequency_hz"] = LINE_FREQUENCY_HZ

        rows.append(cycle_df)
        split_counts = cycle_df["split"].value_counts().to_dict()
        summaries.append((filename, total_cycles, split_counts, int(cycle_df["raw_rows_dropped"].iloc[0])))

    if not rows:
        raise FileNotFoundError("No usable CSV files found. Upload the CSV files before running this cell.")

    rms_df = pd.concat(rows, ignore_index=True)
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    rms_df.to_csv(output_csv, index=False)

    for filename, total_cycles, split_counts, dropped in summaries:
        print(
            f"{filename}: {total_cycles:,} cycles | "
            f"train={split_counts.get('train', 0):,}, val={split_counts.get('val', 0):,}, "
            f"test={split_counts.get('test', 0):,} | dropped_raw_rows={dropped}"
        )

    print(f"\nRMS dataset saved to: {output_csv}")
    print(f"Total RMS cycle rows: {len(rms_df):,}")
    return rms_df

In [4]:
def rotate_sample(sample, phase_class):
    # Original faulty files are Phase 1. Rotating simulates the same fault in Phase 2/3.
    # phase_class 0 -> Phase 1: [I1, I2, I3]
    # phase_class 1 -> Phase 2: [I3, I1, I2]
    # phase_class 2 -> Phase 3: [I2, I3, I1]
    if phase_class == 0:
        return sample
    if phase_class == 1:
        return sample[[2, 0, 1]]
    if phase_class == 2:
        return sample[[1, 2, 0]]
    raise ValueError(f"Invalid phase class: {phase_class}")


def add_features(X):
    X = np.asarray(X, dtype=np.float32)
    i1, i2, i3 = X[:, 0], X[:, 1], X[:, 2]
    eps = 1e-6

    i_sum = i1 + i2 + i3
    i_mean = i_sum / 3.0
    i_abs_max = np.maximum.reduce([np.abs(i1), np.abs(i2), np.abs(i3)])
    i_abs_min = np.minimum.reduce([np.abs(i1), np.abs(i2), np.abs(i3)])
    i_range = i_abs_max - i_abs_min
    i_std = np.std(X, axis=1)
    imbalance_percent = i_range / (np.abs(i_mean) + eps)
    i1_to_mean = i1 / (i_mean + eps)
    i2_to_mean = i2 / (i_mean + eps)
    i3_to_mean = i3 / (i_mean + eps)
    i1_minus_i2 = i1 - i2
    i2_minus_i3 = i2 - i3
    i3_minus_i1 = i3 - i1

    return np.column_stack([
        X,
        i_sum, i_mean, i_abs_max, i_abs_min, i_range, i_std,
        imbalance_percent,
        i1_to_mean, i2_to_mean, i3_to_mean,
        i1_minus_i2, i2_minus_i3, i3_minus_i1,
    ]).astype(np.float32)


def expand_split_for_training(base_df):
    X_raw = base_df[["I1_rms", "I2_rms", "I3_rms"]].to_numpy(dtype=np.float32)
    X_parts = []
    y_binary = []
    y_load = []
    y_severity = []
    y_phase = []

    for row_idx, row in enumerate(base_df.itertuples(index=False)):
        sample = X_raw[row_idx]

        # Rotate healthy rows too for binary/load invariance. Severity/phase are ignored for healthy rows.
        for phase_class in range(3):
            rotated = rotate_sample(sample, phase_class)
            X_parts.append(rotated)
            y_binary.append(int(row.binary))
            y_load.append(int(row.load))

            if int(row.binary) == 1:
                y_severity.append(int(row.severity))
                y_phase.append(phase_class)
            else:
                y_severity.append(-1)
                y_phase.append(-1)

    X = add_features(np.asarray(X_parts, dtype=np.float32))
    y = {
        "binary": np.asarray(y_binary, dtype=np.int32),
        "load": np.asarray(y_load, dtype=np.int32),
        "severity": np.asarray(y_severity, dtype=np.int32),
        "phase": np.asarray(y_phase, dtype=np.int32),
    }
    return X, y


def get_split_data(rms_df, split_name):
    split_df = rms_df[rms_df["split"] == split_name].copy().reset_index(drop=True)
    return expand_split_for_training(split_df)


def make_xgb_classifier(task_name, use_gpu=True):
    params = {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "min_child_weight": 20,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 2.0,
        "random_state": RANDOM_SEED,
        "verbosity": 0,
        "tree_method": "hist",
        "device": "cuda" if use_gpu else "cpu",
    }

    if task_name == "binary":
        params["objective"] = "binary:logistic"
    elif task_name in {"severity", "phase", "load"}:
        params["objective"] = "multi:softprob"
        params["num_class"] = 3
    else:
        raise ValueError(f"Unknown task: {task_name}")

    return xgb.XGBClassifier(**params)


def evaluate_model(name, model, X, y, labels):
    pred = model.predict(X)
    acc = float(accuracy_score(y, pred))
    print(f"{name} accuracy: {acc * 100:.2f}%")
    print("Confusion matrix:")
    print(confusion_matrix(y, pred, labels=list(labels.keys())))
    print(classification_report(y, pred, labels=list(labels.keys()), target_names=list(labels.values()), zero_division=0))
    return acc


def train_models_v2(rms_df, use_gpu=True):
    print("\n" + "=" * 70)
    print("TRAINING V2 MODELS")
    print("=" * 70)

    X_train, y_train = get_split_data(rms_df, "train")
    X_val, y_val = get_split_data(rms_df, "val")
    X_test, y_test = get_split_data(rms_df, "test")

    print(f"Expanded train rows: {len(X_train):,}")
    print(f"Expanded val rows:   {len(X_val):,}")
    print(f"Expanded test rows:  {len(X_test):,}")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    X_val_scaled = scaler.transform(X_val).astype(np.float32)
    X_test_scaled = scaler.transform(X_test).astype(np.float32)

    models = {}
    results = {}

    # Binary and load use all rows.
    for task, labels in [("binary", BINARY_LABELS), ("load", LOAD_LABELS)]:
        print(f"\n--- {task.upper()} ---")
        model = make_xgb_classifier(task, use_gpu=use_gpu)
        t0 = time.time()
        model.fit(X_train_scaled, y_train[task], eval_set=[(X_val_scaled, y_val[task])], verbose=False)
        print(f"Training time: {time.time() - t0:.1f}s")
        val_acc = evaluate_model(f"{task} validation", model, X_val_scaled, y_val[task], labels)
        test_acc = evaluate_model(f"{task} test", model, X_test_scaled, y_test[task], labels)
        models[task] = model
        results[task] = {"val": val_acc, "test": test_acc}

    # Severity and phase are trained only on faulty rows. Healthy rows correctly get N/A at inference.
    faulty_train = y_train["binary"] == 1
    faulty_val = y_val["binary"] == 1
    faulty_test = y_test["binary"] == 1

    for task, labels in [("severity", SEVERITY_LABELS), ("phase", PHASE_LABELS)]:
        print(f"\n--- {task.upper()} FAULTY-ONLY ---")
        model = make_xgb_classifier(task, use_gpu=use_gpu)
        t0 = time.time()
        model.fit(
            X_train_scaled[faulty_train],
            y_train[task][faulty_train],
            eval_set=[(X_val_scaled[faulty_val], y_val[task][faulty_val])],
            verbose=False,
        )
        print(f"Training time: {time.time() - t0:.1f}s")
        val_acc = evaluate_model(f"{task} validation", model, X_val_scaled[faulty_val], y_val[task][faulty_val], labels)
        test_acc = evaluate_model(f"{task} test", model, X_test_scaled[faulty_test], y_test[task][faulty_test], labels)
        models[task] = model
        results[task] = {"val": val_acc, "test": test_acc}

    return models, scaler, results

In [5]:
def save_models_v2(models, scaler, results, output_dir=MODEL_DIR):
    os.makedirs(output_dir, exist_ok=True)
    joblib.dump(scaler, f"{output_dir}/scaler.joblib")
    for name, model in models.items():
        model.save_model(f"{output_dir}/model_{name}.json")
        joblib.dump(model, f"{output_dir}/model_{name}.joblib")

    metadata = {
        "version": "v2",
        "samples_per_cycle": SAMPLES_PER_CYCLE,
        "sample_interval_s": SAMPLE_INTERVAL_S,
        "line_frequency_hz": LINE_FREQUENCY_HZ,
        "train_fraction": TRAIN_FRACTION,
        "val_fraction": VAL_FRACTION,
        "test_fraction": TEST_FRACTION,
        "feature_names": FEATURE_NAMES,
        "binary_labels": BINARY_LABELS,
        "severity_labels": SEVERITY_LABELS,
        "phase_labels": PHASE_LABELS,
        "load_labels": LOAD_LABELS,
        "results": results,
    }
    with open(f"{output_dir}/metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"\nModels saved to: {output_dir}/")


use_gpu = check_gpu()
existing = [f for f in CSV_FILES if os.path.exists(f)]
print(f"\nFound {len(existing)}/{len(CSV_FILES)} CSV files")
if len(existing) == 0:
    raise FileNotFoundError("Upload the CSV files to Colab before training.")

rms_df = build_rms_dataset(existing)
models, scaler, results = train_models_v2(rms_df, use_gpu=use_gpu)
save_models_v2(models, scaler, results)

print("\n" + "=" * 70)
print("V2 SUMMARY")
print("=" * 70)
for task in ["binary", "severity", "phase", "load"]:
    print(f"{task:<10} val={results[task]['val'] * 100:6.2f}% | test={results[task]['test'] * 100:6.2f}%")

rms_df.head()

GPU detected and working

Found 12/12 CSV files

BUILDING FIXED-CYCLE RMS DATASET


Converting raw CSVs to RMS cycles:   0%|          | 0/12 [00:00<?, ?it/s]

Fullload_1_mu_rf3.csv: 1,000 cycles | train=700, val=150, test=150 | dropped_raw_rows=0
Fullload_3_mu_Rf3.csv: 1,000 cycles | train=700, val=150, test=150 | dropped_raw_rows=0
Fullload_5_mu_Rf3.csv: 1,000 cycles | train=700, val=150, test=150 | dropped_raw_rows=0
Fullload_healthy.csv: 1,000 cycles | train=700, val=150, test=150 | dropped_raw_rows=0
Halfload_1_mu_Rf3.csv: 1,000 cycles | train=700, val=150, test=150 | dropped_raw_rows=0
Halfload_3_mu_Rf3.csv: 1,000 cycles | train=700, val=150, test=150 | dropped_raw_rows=0
Halfload_5__rf3.csv: 1,000 cycles | train=700, val=150, test=150 | dropped_raw_rows=0
Halfload_healthy.csv: 1,000 cycles | train=700, val=150, test=150 | dropped_raw_rows=0
Noload_1_mu_Rf3.csv: 1,000 cycles | train=700, val=150, test=150 | dropped_raw_rows=0
Noload_3_mu_Rf3.csv: 1,000 cycles | train=700, val=150, test=150 | dropped_raw_rows=0
Noload_5_mu_Rf3.csv: 1,000 cycles | train=700, val=150, test=150 | dropped_raw_rows=0
Noload_healthy.csv: 1,000 cycles | train=7

,I1_rms,I2_rms,I3_rms,cycle_index,raw_rows_used,raw_rows_dropped,source_file,binary,severity,load,phase,split,samples_per_cycle,sample_interval_s,line_frequency_hz
0,1.758211,1.710003,1.740000,0,1000000,0,Fullload_1_mu_rf3.csv,1,0,2,-1,train,1000,0.00002,50.0
1,1.737056,1.755879,1.754459,1,1000000,0,Fullload_1_mu_rf3.csv,1,0,2,-1,train,1000,0.00002,50.0
2,1.775897,1.718151,1.770994,2,1000000,0,Fullload_1_mu_rf3.csv,1,0,2,-1,train,1000,0.00002,50.0
3,1.726389,1.737007,1.726535,3,1000000,0,Fullload_1_mu_rf3.csv,1,0,2,-1,train,1000,0.00002,50.0
4,1.777734,1.742260,1.798517,4,1000000,0,Fullload_1_mu_rf3.csv,1,0,2,-1,train,1000,0.00002,50.0


## Single Prediction with Confidence

Use this for testing one RMS input. Severity and phase are only returned when the binary model predicts `Faulty` confidently.

In [9]:
def load_models_v2(model_dir=MODEL_DIR):
    scaler = joblib.load(f"{model_dir}/scaler.joblib")
    models = {
        "binary": joblib.load(f"{model_dir}/model_binary.joblib"),
        "severity": joblib.load(f"{model_dir}/model_severity.joblib"),
        "phase": joblib.load(f"{model_dir}/model_phase.joblib"),
        "load": joblib.load(f"{model_dir}/model_load.joblib"),
    }
    return scaler, models


def _predict_label_and_confidence(model, X_scaled, labels):
    proba = model.predict_proba(X_scaled)[0]
    pred_class = int(np.argmax(proba))
    confidence = float(proba[pred_class])
    return labels[pred_class], confidence, proba


def predict_one_rms(i1_rms, i2_rms, i3_rms, scaler, models, confidence_threshold=0.60, max_safe_current=None):
    currents = np.array([i1_rms, i2_rms, i3_rms], dtype=np.float32)

    # Immediate safety rule. Do not wait for ML if current is beyond your chosen safe limit.
    if max_safe_current is not None and np.max(np.abs(currents)) > max_safe_current:
        return {
            "safety_status": "TRIP_IMMEDIATE_OVERCURRENT",
            "binary": "Faulty",
            "binary_confidence": 1.0,
            "severity": "Unknown",
            "severity_confidence": None,
            "phase": "Unknown",
            "phase_confidence": None,
            "load": "Unknown",
            "load_confidence": None,
        }

    X = add_features(currents.reshape(1, -1))
    X_scaled = scaler.transform(X)

    binary_label, binary_conf, _ = _predict_label_and_confidence(models["binary"], X_scaled, BINARY_LABELS)
    load_label, load_conf, _ = _predict_label_and_confidence(models["load"], X_scaled, LOAD_LABELS)

    if binary_conf < confidence_threshold:
        return {
            "safety_status": "ML_UNCERTAIN",
            "binary": "Uncertain",
            "binary_confidence": binary_conf,
            "severity": "N/A",
            "severity_confidence": None,
            "phase": "N/A",
            "phase_confidence": None,
            "load": load_label,
            "load_confidence": load_conf,
        }

    if binary_label == "Healthy":
        severity_label, severity_conf = "N/A", None
        phase_label, phase_conf = "N/A", None
    else:
        severity_label, severity_conf, _ = _predict_label_and_confidence(models["severity"], X_scaled, SEVERITY_LABELS)
        phase_label, phase_conf, _ = _predict_label_and_confidence(models["phase"], X_scaled, PHASE_LABELS)
        if severity_conf < confidence_threshold:
            severity_label = "Uncertain"
        if phase_conf < confidence_threshold:
            phase_label = "Uncertain"

    return {
        "safety_status": "OK",
        "binary": binary_label,
        "binary_confidence": binary_conf,
        "severity": severity_label,
        "severity_confidence": severity_conf,
        "phase": phase_label,
        "phase_confidence": phase_conf,
        "load": load_label,
        "load_confidence": load_conf,
    }


scaler, models = load_models_v2()

# Example from Fullload_1_mu_rf3.csv, first cycle. Expected source label: Faulty, 1u, Phase 1, Full Load.
example = predict_one_rms(1.456810, 1.428707, 1.404529, scaler, models, confidence_threshold=0.60)
example

{'safety_status': 'OK',
 'binary': 'Faulty',
 'binary_confidence': 0.9985617995262146,
 'severity': '5u',
 'severity_confidence': 0.6152158379554749,
 'phase': 'Phase 1',
 'phase_confidence': 0.6758118867874146,
 'load': 'No Load',
 'load_confidence': 0.998790442943573}

## Rolling Decision Helper for Raspberry Pi

This does not wait silently. It gives a prediction every reading, but also keeps the last few predictions and reports a more stable rolling decision. Immediate overcurrent logic should still trip instantly.

In [7]:
class RollingMotorDecision:
    def __init__(self, scaler, models, window_size=5, fault_votes_required=3, confidence_threshold=0.60, max_safe_current=None):
        self.scaler = scaler
        self.models = models
        self.window_size = window_size
        self.fault_votes_required = fault_votes_required
        self.confidence_threshold = confidence_threshold
        self.max_safe_current = max_safe_current
        self.history = deque(maxlen=window_size)

    def update(self, i1_rms, i2_rms, i3_rms):
        current_prediction = predict_one_rms(
            i1_rms,
            i2_rms,
            i3_rms,
            self.scaler,
            self.models,
            confidence_threshold=self.confidence_threshold,
            max_safe_current=self.max_safe_current,
        )

        if current_prediction["safety_status"] == "TRIP_IMMEDIATE_OVERCURRENT":
            return {"current_prediction": current_prediction, "rolling_decision": current_prediction}

        self.history.append(current_prediction)
        fault_votes = sum(1 for item in self.history if item["binary"] == "Faulty")
        healthy_votes = sum(1 for item in self.history if item["binary"] == "Healthy")

        if fault_votes >= self.fault_votes_required:
            binary = "Faulty"
        elif len(self.history) == self.window_size and healthy_votes > fault_votes:
            binary = "Healthy"
        else:
            binary = current_prediction["binary"]

        def majority_value(key):
            values = [item[key] for item in self.history if item[key] not in {"N/A", "Uncertain", "Unknown"}]
            if not values:
                return "N/A"
            return Counter(values).most_common(1)[0][0]

        rolling_decision = {
            "binary": binary,
            "severity": majority_value("severity") if binary == "Faulty" else "N/A",
            "phase": majority_value("phase") if binary == "Faulty" else "N/A",
            "load": majority_value("load"),
            "window_filled": len(self.history),
            "fault_votes": fault_votes,
            "healthy_votes": healthy_votes,
        }

        return {"current_prediction": current_prediction, "rolling_decision": rolling_decision}


rolling = RollingMotorDecision(scaler, models, window_size=5, fault_votes_required=3, confidence_threshold=0.60)

# Demo with five consecutive known faulty-like readings from the same example.
for _ in range(5):
    result = rolling.update(1.758212, 1.710005, 1.740001)

result

{'current_prediction': {'safety_status': 'OK',
  'binary': 'Faulty',
  'binary_confidence': 0.9975281357765198,
  'severity': 'Uncertain',
  'severity_confidence': 0.5828895568847656,
  'phase': 'Phase 1',
  'phase_confidence': 0.9993632435798645,
  'load': 'Full Load',
  'load_confidence': 0.998683750629425},
 'rolling_decision': {'binary': 'Faulty',
  'severity': 'N/A',
  'phase': 'Phase 1',
  'load': 'Full Load',
  'window_filled': 5,
  'fault_votes': 5,
  'healthy_votes': 0}}

In [8]:
from google.colab import files

!zip -r trained_models_v2.zip trained_models_v2 processed_data_v2
files.download('trained_models_v2.zip')

  adding: trained_models_v2/ (stored 0%)
  adding: trained_models_v2/model_severity.json (deflated 72%)
  adding: trained_models_v2/model_load.json (deflated 89%)
  adding: trained_models_v2/model_binary.json (deflated 72%)
  adding: trained_models_v2/model_severity.joblib (deflated 69%)
  adding: trained_models_v2/model_binary.joblib (deflated 69%)
  adding: trained_models_v2/model_phase.joblib (deflated 69%)
  adding: trained_models_v2/metadata.json (deflated 62%)
  adding: trained_models_v2/model_load.joblib (deflated 92%)
  adding: trained_models_v2/model_phase.json (deflated 72%)
  adding: trained_models_v2/scaler.joblib (deflated 29%)
  adding: processed_data_v2/ (stored 0%)
  adding: processed_data_v2/rms_cycle_dataset_v2.csv (deflated 83%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>